# Exploratory Data Analysis (EDA) - Titanic Dataset

This notebook performs comprehensive exploratory data analysis on the Titanic dataset to understand:
- Data quality and missing values
- Feature distributions
- Relationships between features and survival
- Key insights for feature engineering

**Dataset:** Titanic passenger survival prediction
**Source:** HuggingFace (paulopontesm/titanic)
**Target:** Survived (0 = No, 1 = Yes)


In [ ]:
# Standard imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Project imports
from src.data.loader import TitanicDataLoader
from src.config import config

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Imports successful")
print(f"✓ Project: {config.GCP_PROJECT_ID}")
print(f"✓ Region: {config.GCP_REGION}")


## 1. Load Data

Load the Titanic dataset using our reusable data loader.


In [ ]:
# Load data using our reusable loader
loader = TitanicDataLoader()
train_df, test_df = loader.load_data()

# Display dataset information
loader.display_info()

# Store for analysis (we'll focus on training data for EDA)
df = train_df.copy()

print(f"\n✓ Data loaded successfully")
print(f"  Training set shape: {train_df.shape}")
print(f"  Test set shape: {test_df.shape if test_df is not None else 'N/A'}")


## 2. Basic Statistics

Examine the structure and basic statistics of the dataset.


In [ ]:
# Display first few rows
print("First 5 rows:")
print("="*70)
df.head()


In [ ]:
# Data types and null counts
print("Data Types and Missing Values:")
print("="*70)
df.info()


In [ ]:
# Summary statistics for numerical features
print("Summary Statistics:")
print("="*70)
df.describe()


In [ ]:
# Class balance of Survived column
print("Survival Class Distribution:")
print("="*70)
survival_counts = df['Survived'].value_counts()
survival_pct = df['Survived'].value_counts(normalize=True) * 100

print(f"\nCounts:")
print(survival_counts)
print(f"\nPercentages:")
print(survival_pct)

print(f"\n✓ Survival Rate: {df['Survived'].mean():.2%}")
print(f"✓ Class Balance: {'Balanced' if abs(survival_pct[0] - survival_pct[1]) < 10 else 'Imbalanced'}")

# Visualize class distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

survival_counts.plot(kind='bar', ax=ax[0], color=['#e74c3c', '#2ecc71'])
ax[0].set_title('Survival Counts', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Survived (0=No, 1=Yes)')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['Did Not Survive', 'Survived'], rotation=0)

survival_pct.plot(kind='bar', ax=ax[1], color=['#e74c3c', '#2ecc71'])
ax[1].set_title('Survival Percentages', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Survived (0=No, 1=Yes)')
ax[1].set_ylabel('Percentage (%)')
ax[1].set_xticklabels(['Did Not Survive', 'Survived'], rotation=0)
ax[1].set_ylim([0, 100])

plt.tight_layout()
plt.show()


## 3. Missing Data Analysis

Identify and visualize missing values in the dataset.
    

In [ ]:
# Calculate missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
}).sort_values('Missing_Percentage', ascending=False)

print("Missing Data Summary:")
print("="*70)
print(missing_data[missing_data['Missing_Count'] > 0])

# Visualize missing data with heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis')
plt.title('Missing Data Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Bar chart of missing percentages
fig, ax = plt.subplots(figsize=(10, 6))
missing_cols = missing_data[missing_data['Missing_Count'] > 0]
ax.barh(missing_cols['Column'], missing_cols['Missing_Percentage'], color='#e74c3c')
ax.set_xlabel('Missing Percentage (%)', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.set_xlim([0, 100])
for i, v in enumerate(missing_cols['Missing_Percentage']):
    ax.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()


## 4. Feature Distributions

Analyze distributions of numerical and categorical features, and their relationship with survival.


In [ ]:
# Numerical features distributions
numerical_features = ['Age', 'Fare', 'SibSp', 'Parch']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_features):
    axes[idx].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.2f}')
    axes[idx].legend()

plt.tight_layout()
plt.show()

# Categorical features distributions
categorical_features = ['Sex', 'Pclass', 'Embarked']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(categorical_features):
    df[col].value_counts().plot(kind='bar', ax=axes[idx], color='coral', edgecolor='black')
    axes[idx].set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Survival rates by feature
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sex vs Survival
sex_survival = df.groupby('Sex')['Survived'].agg(['mean', 'count'])
axes[0, 0].bar(sex_survival.index, sex_survival['mean'], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0, 0].set_title('Survival Rate by Sex', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Survival Rate')
axes[0, 0].set_ylim([0, 1])
for i, (idx, row) in enumerate(sex_survival.iterrows()):
    axes[0, 0].text(i, row['mean'] + 0.05, f"{row['mean']:.2%}\n(n={row['count']})", 
                    ha='center', fontsize=10, fontweight='bold')

# Pclass vs Survival
pclass_survival = df.groupby('Pclass')['Survived'].agg(['mean', 'count'])
axes[0, 1].bar(pclass_survival.index.astype(str), pclass_survival['mean'], 
               color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='black')
axes[0, 1].set_title('Survival Rate by Passenger Class', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Pclass')
axes[0, 1].set_ylabel('Survival Rate')
axes[0, 1].set_ylim([0, 1])
for i, (idx, row) in enumerate(pclass_survival.iterrows()):
    axes[0, 1].text(i, row['mean'] + 0.05, f"{row['mean']:.2%}\n(n={row['count']})", 
                    ha='center', fontsize=10, fontweight='bold')

# Embarked vs Survival
embarked_survival = df.groupby('Embarked')['Survived'].agg(['mean', 'count'])
axes[1, 0].bar(embarked_survival.index, embarked_survival['mean'], 
               color=['#9b59b6', '#1abc9c', '#34495e'], edgecolor='black')
axes[1, 0].set_title('Survival Rate by Embarkation Port', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Embarked')
axes[1, 0].set_ylabel('Survival Rate')
axes[1, 0].set_ylim([0, 1])
for i, (idx, row) in enumerate(embarked_survival.iterrows()):
    axes[1, 0].text(i, row['mean'] + 0.05, f"{row['mean']:.2%}\n(n={row['count']})", 
                    ha='center', fontsize=10, fontweight='bold')

# Age distribution by survival
axes[1, 1].hist([df[df['Survived']==0]['Age'].dropna(), df[df['Survived']==1]['Age'].dropna()], 
                bins=20, label=['Did Not Survive', 'Survived'], color=['#e74c3c', '#2ecc71'], 
                edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Age Distribution by Survival', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Age')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Print key insights
print("\nKey Insights:")
print("="*70)
print(f"✓ Female survival rate: {df[df['Sex']=='female']['Survived'].mean():.2%}")
print(f"✓ Male survival rate: {df[df['Sex']=='male']['Survived'].mean():.2%}")
print(f"✓ 1st class survival rate: {df[df['Pclass']==1]['Survived'].mean():.2%}")
print(f"✓ 3rd class survival rate: {df[df['Pclass']==3]['Survived'].mean():.2%}")


## 5. Correlation Analysis

Examine correlations between numerical features and the target variable.


In [ ]:
# Create correlation matrix for numerical features
numerical_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
correlation_matrix = df[numerical_cols].corr()

# Visualize correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with Survived (target)
print("\nCorrelation with Survived (Target Variable):")
print("="*70)
survived_corr = correlation_matrix['Survived'].sort_values(ascending=False)
print(survived_corr)

print("\n✓ Strongest positive correlations with survival: Fare")
print("✓ Strongest negative correlations with survival: Pclass")
print("✓ Sex encoding would show strongest correlation (to be added in preprocessing)")


## 6. Summary of Key Findings

**Data Quality:**
- 891 training samples
- Age: 19.9% missing - will need imputation
- Cabin: 77.1% missing - consider dropping or creating binary feature
- Embarked: 0.2% missing - can impute with mode

**Target Variable:**
- 38.38% survival rate (imbalanced classes)
- 549 did not survive, 342 survived

**Key Predictors:**
1. **Sex**: Strongest predictor - women had 74% survival rate vs men at 19%
2. **Pclass**: 1st class (63% survival) >> 3rd class (24% survival)
3. **Fare**: Positively correlated with survival (higher fare = higher class)
4. **Age**: Children had higher survival rates

**Feature Engineering Opportunities:**
- Extract title from Name (Mr., Mrs., Miss., Master., etc.)
- Create family size feature (SibSp + Parch + 1)
- Create "Is_Alone" binary feature
- Bin continuous features (Age, Fare)
- Extract cabin deck letter
- One-hot encode categorical variables

**Next Steps:**
- Handle missing values with appropriate strategies
- Feature engineering to create derived features
- Encode categorical variables
- Feature scaling for numerical features


Wha